In [ ]:
!pip install astropy
!git clone https://github.com/dodo47/GCDetection

In [ ]:
from ipywidgets import interact, IntSlider
from IPython.display import display
import numpy as np
from astropy.io import fits
from pathlib import Path
from matplotlib import pyplot as plt

def plot_image(frame_index, channel_index, fits_index):
    """Plots a FITS file with given frame, channel and fits indexes"""
    with fits.open(fits_files[fits_index]) as fits_file:
      image_data = fits_file[1].data
    single_image = image_data[frame_index, channel_index, :, :]

    plt.figure(figsize=(6, 6))

    image = plt.imshow(
        single_image,
        cmap='viridis',
        origin='lower'
    )

    plt.colorbar(image, label='Pixel Value')
    plt.title(f"{fits_files[fits_index].name}, Frame: {frame_index}, Channel: {channel_index}")
    plt.xlabel("X-pixels")
    plt.ylabel("Y-pixels")
    plt.show()

fits_files = [elem for elem in Path("GCDetection/data/ImageData/").iterdir() if elem.is_file()]
with fits.open(fits_files[0]) as sample:
  dims = (sample[1].shape[0], sample[1].shape[1],len(fits_files))


frame_index=IntSlider(
    min=0,
    max= dims[0] - 1,
    step=1,
    value=0,
    description='Frame Index (630 total):',
    continuous_update=True
)

channel_index=IntSlider(
    min=0,
    max= dims[1] - 1,
    step=1,
    value=0,
    description='Channel Index (2 total):',
    continuous_update=True
)

fits_index=IntSlider(
    min=0,
    max= dims[2] - 1,
    step=1,
    value=0,
    description=f"Image Index ({len(fits_files)} total)",
    continuous_update=True
)


def updater(change):
  new_val = change["new"]
  with fits.open(fits_files[change["new"]]) as new_fits:
    channel_index.max = new_fits[1].shape[1] - 1
    frame_index.max = new_fits[1].shape[0] - 1

fits_index.observe(updater, names="value")
interact(
    plot_image,
    frame_index=frame_index,
    channel_index=channel_index,
    fits_index=fits_index
);